**Assignment 2 - Data cleaning**

**Information about the group and report**

Group Number: 58 \\
Name of Team Members: Yasmine Zoubdi and Anoushka Jawale \\
Student Numbers : 300170464 and 300233148 \\

In [ ]:
!pip install nbimporter

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import r2_score
import nbimporter
from CSI4142-Assignment2.ipynb import *



SyntaxError: invalid syntax (<ipython-input-4-123f4f7b8385>, line 12)

##### **Part 1: Validity checker**

#####**First look at data:**

In [ ]:
url = "https://raw.githubusercontent.com/anoushka-j/netflix_shows_data/refs/heads/main/netflix_titles.csv"
data = pd.read_csv(url)
data.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


**a) Description of the Dataset**

The dataset contains all TV Shows and Movies meta data on Netflix, and is updated every month.




**b) Data Dictionary**

 The dataset contains the following features, often collected by medical professionals to analyze patient's conditions as they relate to heart disease. The following table contains the descriptions of all features found in the dataset.

In [ ]:
print(data.columns)

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')


| **Feature Name**   | **Description**                                                                 | **Data Type**     |
|--------------------|---------------------------------------------------------------------------------|-------------------|
| `show_id`          | Unique identifier for each show or movie in the dataset.                         | `object` (String) |
| `type`             | Type of the show or movie (e.g., Movie, TV Show).                               | `object` (String) |
| `title`            | Title of the movie or TV show.                                                   | `object` (String) |
| `director`         | Name of the director(s) of the movie or TV show.                                | `object` (String) |
| `cast`             | List of actors or actresses in the movie or TV show.                             | `object` (String) |
| `country`          | Country where the show or movie was produced or primarily available.            | `object` (String) |
| `date_added`       | The date when the show or movie was added to Netflix.                           | `object` (String) |
| `release_year`     | The year the movie or TV show was released.                                     | `int64` (Integer) |
| `rating`           | The age rating of the show or movie (e.g., PG-13, TV-MA).                        | `object` (String) |
| `duration`         | The length of the movie (in minutes) or the number of seasons of a TV show.     | `object` (String) |
| `listed_in`        | Categories or genres the show or movie falls under.                             | `object` (String) |
| `description`      | A brief summary of the show or movie's plot or theme.                           | `object` (String) |


##### Dataset Info:
From below it can be seen there are some null values in the dataset, especially in the director, cast, and country column. But this is only a first look.

In [ ]:
data.info()
data.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


**c) Clean data checker**

###### **1) Data Type errors**

In this test, we will verify the data type of any column given the name of the column.

**Parameters for the checker:**

In [ ]:
expected_data_types = {
    'show_id': 'object',  # String (identifier)
    'type': 'object',  # String (Movie or TV Show)
    'title': 'object',  # String (Title of the movie or TV show)
    'director': 'object',  # String (Director(s) of the show)
    'cast': 'object',  # String (List of actors or actresses)
    'country': 'object',  # String (Country of production)
    'date_added': 'object',  # Date (Added to Netflix)
    'release_year': 'int64',  # Integer (Release year of the show)
    'rating': 'object',  # String (Age rating like PG, TV-MA, etc.)
    'duration': 'object',  # String (Duration or number of seasons)
    'listed_in': 'object',  # String (Genres or categories)
    'description': 'object'  # String (Description of the show)
}

attributes = data.columns
test_attribute = "release_year"
test_attribute_dtype = data[test_attribute].dtype


**Checker code:**

In [ ]:
def check_data_type(data, test_attribute):
    actual_type = data[test_attribute].dtype
    if actual_type != expected_data_types[test_attribute]:
        return f"Error: {test_attribute} should be of type {expected_data_types[test_attribute]}, but is {actual_type}."
    return f"'{test_attribute}' data type is found to be {actual_type}, which was as expected."

check_data_type(data, test_attribute)

"'release_year' data type is found to be int64, which was as expected."

**Report of findings:**

###### **2) Range errors**

In this test, we will verify the range of a numerical value. The range is the minimum and maximum
values that an attribute can have.

**Parameters for the checker:**

In [ ]:
# Attribute selection
test_attribute = 'release_year'
minimum = 1980
maximum = 2000

**Checker code:**

In [ ]:
def range_check(df, test_attribute, min_value, max_value):

    # Count values less than the min_value
    less_than_min_rows = df[df[test_attribute] < min_value]

    # Store rows that are greater than the max_value
    greater_than_max_rows = df[df[test_attribute] > max_value]

    # Count how many values are less than the min_value and greater than the max_value
    less_than_min = len(less_than_min_rows)
    greater_than_max = len(greater_than_max_rows)

    # Output the full report
    print(f"There are {less_than_min} data points with {test_attribute} less than {min_value}, and {greater_than_max} data points with {test_attribute} over {max_value}.")
    print(f"See for example the following rows, noting the {test_attribute}:\n")

    print(less_than_min_rows.iloc[0])
    print("_____________________________")
    print(greater_than_max_rows.iloc[0])
    return f"Checked {test_attribute}: {less_than_min} values less than {min_value} and {greater_than_max} values greater than {max_value}."



**Report of findings:**

In [ ]:
range_check(data, test_attribute, minimum, maximum)


There are 122 data points with release_year less than 1980, and 8245 data points with release_year over 2000.
See for example the following rows, noting the release_year:

show_id                                                       s42
type                                                        Movie
title                                                        Jaws
director                                         Steven Spielberg
cast            Roy Scheider, Robert Shaw, Richard Dreyfuss, L...
country                                             United States
date_added                                     September 16, 2021
release_year                                                 1975
rating                                                         PG
duration                                                  124 min
listed_in              Action & Adventure, Classic Movies, Dramas
description     When an insatiable great white shark terrorize...
Name: 41, dtype: object
____________

'Checked release_year: 122 values less than 1980 and 8245 values greater than 2000.'

###### **3) Format errors**

A format check ensures the data of a particular column is in the expected format, following set rules.

**Parameters for the checker:**

In [ ]:
expected_formats = {
    'date_added': r'([A-Za-z]+)\s*(\d{1,2})\s*,\s*(\d{4})',  # Date format: YYYY-MM-DD
    'duration': r'^\d+\s(min|Season|Seasons)$',  # Duration format: e.g., "90 min" or "1 Season"
    'rating': r'^(G|PG|PG-13|R|TV-[A-Za-z0-9-]+|NC-[A-Za-z0-9-]+)$',  # Valid rating formats
    'title': r'.+',  # Title should not be empty
}
attributes = ['date_added', 'duration', 'rating', 'title']
test_attribute = "rating"
test_attribute_format = expected_formats[test_attribute]


**Checker code:**

In [ ]:
def check_format_errors(df, test_attribute):

    pattern = expected_formats[test_attribute]

    df[test_attribute] = df[test_attribute].astype(str)

    # Apply the format check using regex
    invalid_data = df[~df[test_attribute].str.contains(pattern, na=False)]

    if not invalid_data.empty:
        # Append error message with a sample of invalid data
        print(f"There are {invalid_data.shape[0]} rows with invalid format in '{test_attribute}'.")
        print(f"See for example the following row, noting the '{test_attribute}':")
        print(invalid_data.iloc[0])
        print(invalid_data.iloc[1])


    else:
        # If no errors, print success message
        print(f"All rows in column '{test_attribute}' match the expected format.")




**Report of findings:**

In [ ]:
check_format_errors(data, test_attribute)

There are 90 rows with invalid format in 'rating'.
See for example the following row, noting the 'rating':
show_id                                                     s5542
type                                                        Movie
title                                             Louis C.K. 2017
director                                               Louis C.K.
cast                                                   Louis C.K.
country                                             United States
date_added                                          April 4, 2017
release_year                                                 2017
rating                                                     74 min
duration                                                      nan
listed_in                                                  Movies
description     Louis C.K. muses on religion, eternal love, gi...
Name: 5541, dtype: object
show_id                                                     s5795
type     

<ipython-input-94-70ad5270ba26>:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  invalid_data = df[~df[test_attribute].str.contains(pattern, na=False)]


###### **4) Consistency errors**

This check tests whether values in the dataset are consistent with each other. For this dataset specifically I will focus on the columns of release_date and date)added, to ensure they the date_added is after the release year.

**Parameters for the checker:**

In [ ]:
test_attributes = ['release_year', 'date_added']


**Checker code:**

In [ ]:
import pandas as pd

def check_consistency_error(df, test_attributes):

    date_added_column = test_attributes[1]
    release_date_column = test_attributes[0]

    # Convert columns to datetime for comparison
    df[date_added_column] = pd.to_datetime(df[date_added_column], errors='coerce')
    df[release_date_column] = pd.to_datetime(df[release_date_column], errors='coerce')

    # Check if 'date_added' is before 'release_date'
    invalid_data = df[df[date_added_column] < df[release_date_column]]

    if not invalid_data.empty:
        # Report error if any date_added is before release_date
        print(f"There are {invalid_data.shape[0]} rows where 'date_added' is before 'release_date'.")
        print(f"See for example the following rows, noting the '{date_added_column}' and '{release_date_column}':")
        print(invalid_data[[date_added_column, release_date_column]].head(2))  # Displaying the first 2 rows as examples
    else:
        # If no errors, print success message
        print(f"All rows have 'date_added' after 'release_date'.")


check_consistency_error(data, test_attributes)


All rows have 'date_added' after 'release_date'.


**Report of findings:**

###### **5) Uniqueness errors**

In this test, we will verify the uniqueness of the id of the dataset, (the show_id) and check if it is unique.

**Parameters for the checker:**

In [ ]:
attributes = ['show_id']
test_attribute = "show_id"

**Checker code:**

In [ ]:
def check_uniqueness_errors(df, test_attribute):
    non_unique_data = df[df.duplicated(subset=[test_attribute], keep=False)]

    if not non_unique_data.empty:
        # Report how many non-unique values exist
        print(f"There are {non_unique_data.shape[0]} rows with values that are not unique in '{test_attribute}'.")
        print(f"See for example the following rows, noting the '{test_attribute}':")
        print(non_unique_data.iloc[0])  # Print the first row with non-unique value
        print(non_unique_data.iloc[1])  # Print the second row with non-unique value (if available)
    else:
        # If all values are unique, print success message
        print(f"All rows in column '{test_attribute}' are unique.")



**Report of findings:**

In [ ]:
check_uniqueness_errors(data, test_attribute)

There are 5146 rows with values that are not unique in 'director'.
See for example the following rows, noting the 'director':
show_id                                                        s2
type                                                      TV Show
title                                               Blood & Water
director                                                      NaN
cast            Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...
country                                              South Africa
date_added                                     September 24, 2021
release_year                                                 2021
rating                                                      TV-MA
duration                                                2 Seasons
listed_in         International TV Shows, TV Dramas, TV Mysteries
description     After crossing paths at a party, a Cape Town t...
Name: 1, dtype: object
show_id                                                    

###### **6) Presence errors**

In the presence test, we validate the is_nan() counts of the columns shown earlier in the notebook, and check whether there are any NaN or blank values within the required columns.

**Parameters for the checker:**

In [ ]:
attributes = data.columns #assuming all fields are required
test_attribute = "date_added"


**Checker code:**

In [ ]:
import pandas as pd

def check_presence_errors(df, test_attribute):
    # Check for missing values (NaN, None, or empty string)
    missing_data = df[df[test_attribute].isna() |
                      (df[test_attribute].str.strip() == '') |
                      df[test_attribute].str.lower().isin(['nan', 'none', 'na'])]
    if not missing_data.empty:
        # Report how many missing values exist
        print(f"There are {missing_data.shape[0]} rows with missing values in '{test_attribute}'.")
        print(f"See for example the following rows, noting the '{test_attribute}':")
        print(missing_data.iloc[0])  # Print the first row with missing value
        print(missing_data.iloc[1])  # Print the second row with missing value (if available)
    else:
        # If no missing values, print success message
        print(f"All rows in column '{test_attribute}' have valid values (no missing values).")


**Report of findings:**

In [ ]:
check_presence_errors(data, test_attribute)


There are 10 rows with missing values in 'date_added'.
See for example the following rows, noting the 'date_added':
show_id                                                     s6067
type                                                      TV Show
title                 A Young Doctor's Notebook and Other Stories
director                                                      NaN
cast            Daniel Radcliffe, Jon Hamm, Adam Godley, Chris...
country                                            United Kingdom
date_added                                                    nan
release_year                                                 2013
rating                                                      TV-MA
duration                                                2 Seasons
listed_in                British TV Shows, TV Comedies, TV Dramas
description     Set during the Russian Revolution, this comic ...
Name: 6066, dtype: object
show_id                                                     s6175


###### **7) Length errors**

In this test, we check the max lengths on some of the longer string columns in the dataset and provide an upper bound onthe description, for example.

**Parameters for the checker:**

In [ ]:
attributes = ["description", "title", "cast", "director"]
test_attribute = "description"
max_length = 200

**Checker code:**

In [ ]:
import pandas as pd

def check_length_errors(df, test_attribute, max_length):
    length_error_data = df[df[test_attribute].str.len() > max_length]

    if not length_error_data.empty:
        # Report how many length errors exist
        print(f"There are {length_error_data.shape[0]} rows with length errors in '{test_attribute}'.")
        print(f"See for example the following rows, noting the '{test_attribute}':")
        print(length_error_data.iloc[0])  # Print the first row with length error
        print(length_error_data.iloc[1])  # Print the second row with length error (if available)
    else:
        # If no length errors, print success message
        print(f"All rows in column '{test_attribute}' meet the maximum length requirement of {max_length} characters.")



**Report of findings:**

In [ ]:
check_length_errors(data, test_attribute, 200)

There are 32 rows with length errors in 'description'.
See for example the following rows, noting the 'description':
show_id                                                      s216
type                                                        Movie
title                                     Shootout at Lokhandwala
director                                           Apoorva Lakhia
cast            Amitabh Bachchan, Sanjay Dutt, Sunil Shetty, A...
country                                                     India
date_added                                        August 27, 2021
release_year                                                 2007
rating                                                      TV-MA
duration                                                  116 min
listed_in        Action & Adventure, Dramas, International Movies
description     Based on a true story, this action film follow...
Name: 215, dtype: object
show_id                                                      s594


###### **8) Look-up errors**

In this test, check if certain columns that can have look-up values such as rating and listed_in, only contain verified values of a set.

**Parameters for the checker:**

In [ ]:
attributes = ['rating', 'type', 'listed_in']
test_attribute = "listed_in"

**Checker code:**

In [ ]:
import pandas as pd

def check_lookup_errors(df, test_attribute):

    non_blank_values = df[test_attribute].dropna()  # Remove NaN values
    non_blank_values = non_blank_values[non_blank_values != '']  # Remove blank/empty

    # Get unique values from the column
    unique_values = non_blank_values.unique()

    # Check for values that are not in the valid list of values
    lookup_error_data = df[~df[test_attribute].isin(unique_values)]

    if not lookup_error_data.empty:
        # Report how many lookup errors exist
        print(f"There are {lookup_error_data.shape[0]} rows with lookup errors in '{test_attribute}'.")
        print(f"See for example the following rows, noting the '{test_attribute}':")
        print(lookup_error_data.iloc[0])  # Print the first row with lookup error
        print(lookup_error_data.iloc[1])  # Print the second row with lookup error (if available)
    else:
        # If no lookup errors, print success message
        print(f"All rows in column '{test_attribute}' match the valid lookup values.")



**Report of findings:**

In [ ]:
check_lookup_errors(data, test_attribute)

All rows in column 'listed_in' match the valid lookup values.


###### **9) Exact duplicate errors**

This test finds the exact duplicates within the data, so the entire dataset is passed into the function.

**Parameters for the checker:**

In [ ]:
#function takes all data to check for exact duplicates

**Checker code:**

In [ ]:
def exact_duplicate_check(df):
    # Identify exact duplicate rows
    duplicate_rows = df[df.duplicated(keep=False)]  # `keep=False` marks all duplicates

    # Count the number of duplicate rows
    duplicate_count = len(duplicate_rows)

    # Output the full report
    print(f"There are {duplicate_count} exact duplicate rows.")

    if duplicate_count > 0:
        print(f"See for example the following duplicate rows:\n")
        print(duplicate_rows.iloc[0])  # Show first duplicate row
        print("_____________________________")
        print(duplicate_rows.iloc[1])  # Show second duplicate row (if available)

exact_duplicate_check(data)


There are 0 exact duplicate rows.


**Report of findings:**

###### **10) Near duplicate errors**

In this test, find the near duplicates within the dataset, based on a subset of columns.

**Parameters for the checker:**

In [ ]:
attributes = data.columns
test_attribute = ['title', 'description']

**Checker code:**

In [ ]:

def normalize_string(s):

    #converting to lowercase and stripping extra spaces.

    return " ".join(s.strip().lower().split())

def find_near_duplicates(df, columns, threshold=40):
    #Find near duplicates based on a set of columns by comparing normalized strings through helper function above
    near_duplicates = []

    # Combine the selected columns into a single string for each row
    df['combined'] = df[columns].astype(str).agg(' '.join, axis=1)
    df['normalized'] = df['combined'].apply(normalize_string)

    # Compare each normalized string to every other value
    for i, val1 in enumerate(df['normalized']):
        for j, val2 in enumerate(df['normalized']):
          #do not compare same value with itself otherwise will cause false near duplicates
            if i >= j:
                continue
            # If the normalized strings are exactly the same, flag as a near duplicate
            if val1 == val2:
                near_duplicates.append((df.index[i], df.index[j], val1, val2))

    print("Below are the near duplicates found, with the row ids and the match:")
    for x in near_duplicates:
        print(x)
    print("Below is an example of two columns that were found to be near duplicates. Note that they are not EXACT duplicates due to a few fields not matching.")
    print(data.iloc[303])
    print(data.iloc[6705])


**Report of findings:**

In [ ]:
find_near_duplicates(data, test_attribute)

Below are the near duplicates found, with the row ids and the match:
(303, 6705, 'esperando la carroza cora has three sons and a daughter and she´s almost 80. one day during a family reunion the big question comes up: who will be her heir?', 'esperando la carroza cora has three sons and a daughter and she´s almost 80. one day during a family reunion the big question comes up: who will be her heir?')
(1270, 8022, 'sin senos sí hay paraíso born into a small town controlled by the mafia, an irate young woman seeks revenge on the forces that tore apart and wrongfully imprisoned her family.', 'sin senos sí hay paraíso born into a small town controlled by the mafia, an irate young woman seeks revenge on the forces that tore apart and wrongfully imprisoned her family.')
(3371, 6529, 'consequences secrets bubble to the surface after a sensual encounter and an unforeseen crime entangle two friends and a woman caught between them.', 'consequences secrets bubble to the surface after a sensual enc

##### **Conclusion**

The above ten error handling functions help in the process of data cleaning to ensure data integrity and accuracy during analysis.

##### **Part 2: Imputation**

**a) Description of the Dataset**

This dataset is retrieved from [kaggle](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/rules). The dataset comprises 79 explanatory variables describing residential properties in Ames, Iowa, used to predict final sale prices. It includes detailed information on physical attributes (e.g., square footage, basement quality, number of bedrooms), location (e.g., neighborhood, zoning), structural features (e.g., roof type, foundation), condition ratings (e.g., overall quality, exterior condition), and sale details (e.g., sale type, year sold). Key variables like OverallQual (overall material/finish quality), Neighborhood, GrLivArea (above-grade living area), and YearBuilt are critical for understanding housing market trends. The dataset is widely used in regression modeling to predict home prices based on a combination of categorical, ordinal, and numerical features.

**b) Data Dictionnary**

 The dataset contains the following features:

| **Feature**            | **Description**                                                                                                         | **Type**                             |
|------------------------|-------------------------------------------------------------------------------------------------------------------------|--------------------------------------|
| **MSSubClass**         | Dwelling type [20=1-Story 1946+, 30=1-Story 1945-, ..., 190=2-Family Conversion]                                         | Categorical (nominal)               |
| **MSZoning**           | Zoning classification [A=Agriculture, C=Commercial, FV=Floating Village, RL=Residential Low Density, etc.]              | Categorical (nominal)               |
| **LotFrontage**        | Linear feet of street connected to property                                                                             | Numerical                            |
| **LotArea**            | Lot size in square feet                                                                                                 | Numerical                            |
| **Street**             | Road access [Grvl=Gravel, Pave=Paved]                                                                                   | Categorical (binary)                |
| **Alley**              | Alley access [Grvl=Gravel, Pave=Paved, NA=No alley]                                                                     | Categorical (nominal)               |
| **LotShape**           | Property shape [Reg=Regular, IR1-IR3=Increasing irregularity]                                                           | Categorical (ordinal)               |
| **LandContour**        | Flatness [Lvl=Flat, Bnk=Banked, HLS=Hillside, Low=Depression]                                                           | Categorical (nominal)               |
| **Utilities**          | Available utilities [AllPub=All public, NoSewr=Septic tank, NoSeWa=Electric+Gas only, ELO=Electric only]                | Categorical (nominal)               |
| **LotConfig**          | Lot configuration [Inside, Corner, CulDSac, FR2/FR3=Frontage on 2/3 sides]                                              | Categorical (nominal)               |
| **LandSlope**          | Slope [Gtl=Gentle, Mod=Moderate, Sev=Severe]                                                                            | Categorical (ordinal)               |
| **Neighborhood**       | Physical location [25 locations including Blmngtn=Bloomington Heights, NWAmes=NW Ames, etc.]                            | Categorical (nominal)               |
| **Condition1/2**       | Proximity to features [Arterial streets, railroads, parks, etc.]                                                        | Categorical (nominal)               |
| **BldgType**           | Dwelling type [1Fam=Single-family, 2FmCon=Two-family conversion, Duplex, Townhouse]                                     | Categorical (nominal)               |
| **HouseStyle**         | Style [1Story, 1.5Unf/Fin, 2Story, 2.5Unf/Fin, SFoyer=Split Foyer, SLvl=Split Level]                                    | Categorical (nominal)               |
| **OverallQual**        | Overall material/finish quality [1=Very Poor to 10=Very Excellent]                                                      | Ordinal                              |
| **OverallCond**        | Overall condition [1=Very Poor to 10=Very Excellent]                                                                    | Ordinal                              |
| **YearBuilt**          | Original construction year                                                                                             | Numerical                            |
| **YearRemodAdd**       | Remodel year (matches construction if no remodeling)                                                                    | Numerical                            |
| **RoofStyle**          | Roof type [Flat, Gable, Gambrel, Hip, Mansard, Shed]                                                                    | Categorical (nominal)               |
| **RoofMatl**           | Roof material [Clay Tile, Composite Shingle, Metal, Wood Shakes, etc.]                                                  | Categorical (nominal)               |
| **Exterior1st/2nd**    | Exterior covering [Vinyl Siding, Brick, Wood, Stucco, etc.]                                                             | Categorical (nominal)               |
| **MasVnrType**         | Masonry veneer type [Brick, Stone, Cinder Block, None]                                                                  | Categorical (nominal)               |
| **MasVnrArea**         | Masonry veneer area (sq ft)                                                                                             | Numerical                            |
| **ExterQual**          | Exterior quality [Ex=Excellent, Gd=Good, TA=Average, Fa=Fair, Po=Poor]                                                  | Ordinal                              |
| **ExterCond**          | Exterior condition [Same scale as ExterQual]                                                                            | Ordinal                              |
| **Foundation**         | Foundation type [BrkTil=Brick & Tile, CBlock=Cinder Block, PConc=Poured Concrete, etc.]                                 | Categorical (nominal)               |
| **BsmtQual**           | Basement height quality [Ex=Excellent (>100"), ..., Po=Poor (<70"), NA=No basement]                                      | Ordinal                              |
| **BsmtCond**           | Basement condition [Ex=Excellent, ..., Po=Poor, NA=No basement]                                                         | Ordinal                              |
| **BsmtExposure**       | Basement exposure [Gd=Good, Av=Average, Mn=Minimum, No=None, NA=No basement]                                            | Categorical (ordinal)               |
| **BsmtFinType1/2**     | Basement finish quality [GLQ=Good Living Quarters, ..., Unf=Unfinished, NA=No basement]                                 | Categorical (ordinal)               |
| **BsmtFinSF1/2**       | Finished basement area (Type 1/2) in sq ft                                                                              | Numerical                            |
| **BsmtUnfSF**          | Unfinished basement area (sq ft)                                                                                        | Numerical                            |
| **TotalBsmtSF**        | Total basement area (sq ft)                                                                                             | Numerical                            |
| **Heating**            | Heating type [GasA=Gas forced air, GasW=Gas water, Grav=Gravity, etc.]                                                  | Categorical (nominal)               |
| **HeatingQC**          | Heating quality [Same Ex-Po scale as ExterQual]                                                                          | Ordinal                              |
| **CentralAir**         | Central air conditioning [Y=Yes, N=No]                                                                                   | Categorical (binary)                |
| **Electrical**         | Electrical system [SBrkr=Standard Breaker, FuseA/FuseF/FuseP=Fuse Box types, Mix=Mixed]                                 | Categorical (nominal)               |
| **1stFlrSF**           | First floor square feet                                                                                                 | Numerical                            |
| **2ndFlrSF**           | Second floor square feet                                                                                                | Numerical                            |
| **LowQualFinSF**       | Low quality finished area (all floors)                                                                                  | Numerical                            |
| **GrLivArea**          | Above-grade living area (sq ft)                                                                                         | Numerical                            |
| **BsmtFullBath**       | Basement full bathrooms                                                                                                 | Numerical                            |
| **BsmtHalfBath**       | Basement half bathrooms                                                                                                 | Numerical                            |
| **FullBath**           | Full bathrooms above grade                                                                                              | Numerical                            |
| **HalfBath**           | Half bathrooms above grade                                                                                              | Numerical                            |
| **Bedroom**            | Bedrooms above grade                                                                                                    | Numerical                            |
| **Kitchen**            | Kitchens above grade                                                                                                    | Numerical                            |
| **KitchenQual**        | Kitchen quality [Ex=Excellent to Po=Poor]                                                                                | Ordinal                              |
| **TotRmsAbvGrd**       | Total rooms above grade (excluding bathrooms)                                                                           | Numerical                            |
| **Functional**         | Home functionality [Typ=Typical, Min1/Min2=Minor deductions, Mod=Moderate, Maj1/Maj2=Major, Sev=Severe, Sal=Salvage]    | Categorical (ordinal)               |
| **Fireplaces**         | Number of fireplaces                                                                                                    | Numerical                            |
| **FireplaceQu**        | Fireplace quality [Ex=Excellent masonry to Po=Poor stove, NA=No fireplace]                                              | Ordinal                              |
| **GarageType**         | Garage location [Attchd=Attached, Detchd=Detached, BuiltIn, etc., NA=No garage]                                         | Categorical (nominal)               |
| **GarageYrBlt**        | Year garage built                                                                                                       | Numerical                            |
| **GarageFinish**       | Interior finish [Fin=Finished, RFn=Rough Finish, Unf=Unfinished, NA=No garage]                                          | Categorical (ordinal)               |
| **GarageCars**         | Garage car capacity                                                                                                     | Numerical                            |
| **GarageArea**         | Garage area (sq ft)                                                                                                     | Numerical                            |
| **GarageQual**         | Garage quality [Same Ex-Po scale, NA=No garage]                                                                          | Ordinal                              |
| **GarageCond**         | Garage condition [Same Ex-Po scale, NA=No garage]                                                                        | Ordinal                              |
| **PavedDrive**         | Driveway paving [Y=Paved, P=Partial, N=Dirt/Gravel]                                                                     | Categorical (ordinal)               |
| **WoodDeckSF**         | Wood deck area (sq ft)                                                                                                  | Numerical                            |
| **OpenPorchSF**        | Open porch area (sq ft)                                                                                                 | Numerical                            |
| **EnclosedPorch**      | Enclosed porch area (sq ft)                                                                                             | Numerical                            |
| **3SsnPorch**          | Three-season porch area (sq ft)                                                                                         | Numerical                            |
| **ScreenPorch**        | Screen porch area (sq ft)                                                                                               | Numerical                            |
| **PoolArea**           | Pool area (sq ft)                                                                                                       | Numerical                            |
| **PoolQC**             | Pool quality [Ex=Excellent to Fa=Fair, NA=No pool]                                                                      | Ordinal                              |
| **Fence**              | Fence quality [GdPrv=Good Privacy to MnWw=Minimum Wood/Wire, NA=No fence]                                               | Categorical (ordinal)               |
| **MiscFeature**        | Miscellaneous feature [Elev=Elevator, Shed, TenC=Tennis court, NA=None]                                                 | Categorical (nominal)               |
| **MiscVal**            | Value of miscellaneous feature ($)                                                                                      | Numerical                            |
| **MoSold**             | Month sold (1-12)                                                                                                       | Numerical                            |
| **YrSold**             | Year sold (YYYY)                                                                                                        | Numerical                            |
| **SaleType**           | Type of sale [WD=Warranty Deed, New=New Construction, COD=Court Sale, etc.]                                             | Categorical (nominal)               |
| **SaleCondition**      | Sale condition [Normal, Abnorml=Abnormal, Partial=New home partial completion, etc.]                                    | Categorical (nominal)               |


Missing data:

In [ ]:
url = "https://raw.githubusercontent.com/Yasmine08POG/CSI4142/refs/heads/main/house-prices.csv"
df2 = pd.read_csv(url)
missing_table = pd.DataFrame({
    'Missing Values': df2.isnull().sum(),
    'Missing Percentage': (df2.isnull().sum() / len(df2)) * 100
})
pd.set_option('display.max_rows', None)
missing_table

,Missing Values,Missing Percentage
Id,0,0.000000
MSSubClass,0,0.000000
MSZoning,0,0.000000
LotFrontage,259,17.739726
LotArea,0,0.000000
Street,0,0.000000
Alley,1369,93.767123
LotShape,0,0.000000
LandContour,0,0.000000
Utilities,0,0.000000


**Variables removal:** \
 The percentage of missing data for the columns Alley, PoolQC, Fence and MiscFeature is above 80% so we decided to delete these columns in the steps.

In [ ]:
columns_to_drop = missing_table[missing_table['Missing Percentage'] > 80].index
df2 = df2.drop(columns=columns_to_drop)

In the next steps we will perform 3 imputation methods on the MasVnrType, FireplaceQu and LotFrontage variables. We will remove all missing data in the dataset and simulate missing values for these attributes. We will simulate 3 types of missing data; MCAR, MAR and MNAR.

In [ ]:
df2 = df2.dropna()

###### **c) Default value imputation for the variable MasVnrType**

We simulate MCAR for the MasVnrType:

In [ ]:
# Simulate MCAR (Randomly remove 30% of values)
np.random.seed(1)
df2_mcar = df2.copy()
mcar_mask = np.random.rand(len(df2_mcar)) < 0.3
df2_mcar.loc[mcar_mask, 'MasVnrType'] = np.nan
NA_index = df2_mcar["MasVnrType"].isna()
print("MCAR missing values for MasVnrType:", df2_mcar['MasVnrType'].isna().sum())

MCAR missing values for MasVnrType: 88


Mode Imputation:

In [ ]:
# Mode imputation
df2_mcar["MasVnrType"].fillna(df2_mcar["MasVnrType"].mode()[0], inplace=True)

<ipython-input-12-c1db7e41c893>:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df2_mcar["MasVnrType"].fillna(df2_mcar["MasVnrType"].mode()[0], inplace=True)


Evaluation:

In [ ]:
encoder = LabelEncoder()
true_values_encoded = encoder.fit_transform(df2["MasVnrType"][NA_index])
imputed_values_encoded = encoder.transform(df2_mcar["MasVnrType"][NA_index])
r2 = r2_score(true_values_encoded, imputed_values_encoded)
print("R-squared (R²) for the imputation:", r2)

R-squared (R²) for the imputation: -0.37807606263982096


###### **d) Predictive imputation for the FireplaceQu**

We simulate MAR for the FireplaceQu:

In [ ]:
# Simulate MAR (Remove values where 'Fireplaces' is above 1)
df2_mar = df2.copy()
df2_mar.loc[df2_mar["Fireplaces"] > 1, 'FireplaceQu'] = np.nan
NA_index = df2_mar["FireplaceQu"].isna()
print("MAR missing values for FireplaceQu:", df2_mar['FireplaceQu'].isna().sum())

MAR missing values for FireplaceQu: 41


Logistic regression imputation using the Fireplaces variable:

In [ ]:
# Encode FireplaceQu variable to numeric values
encoder = LabelEncoder()
df2_mar["FireplaceQu_encoded"] = encoder.fit_transform(df2_mar["FireplaceQu"].astype(str))

# Separate known and missing values
known = df2_mar[df2_mar["FireplaceQu"].notna()]
missing = df2_mar[df2_mar["FireplaceQu"].isna()]

# Train Logistic Regression model using 'Fireplaces' to predict 'FireplaceQu'
clf = LogisticRegression()
clf.fit(known[["Fireplaces"]], known["FireplaceQu_encoded"])

# Predict missing values
predicted_values = clf.predict(missing[["Fireplaces"]])

# Convert predictions back to original categories
df2_mar.loc[df2_mar["FireplaceQu"].isna(), "FireplaceQu"] = encoder.inverse_transform(predicted_values)
df2_mar.drop(columns=["FireplaceQu_encoded"], inplace=True)

Evaluation:

In [ ]:
# Calculate R² score
encoder = LabelEncoder()
true_values_encoded = encoder.fit_transform(df2["FireplaceQu"][NA_index])
imputed_values_encoded = encoder.transform(df2_mar["FireplaceQu"][NA_index])
r2 = r2_score(true_values_encoded, imputed_values_encoded)
print("R-squared (R²) for the imputation:", r2)

R-squared (R²) for the imputation: -0.02310536044362288


###### **e) KNN imputation for the LotFrontage** **texte en gras**

We simulate MNAR for the LotFrontage:

In [ ]:
# Simulate MNAR by removing LotFrontage values if they are above 100
threshold = 100
df2_mnar = df2.copy()
df2_mnar.loc[df2_mnar["LotFrontage"] > threshold, "LotFrontage"] = np.nan

KNN imputation:

In [ ]:
X = df2_mnar.drop(columns=["LotFrontage"])

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = X.select_dtypes(include=np.number).columns.tolist()

# One-hot encode categoricals
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_encoded = pd.DataFrame(
    encoder.fit_transform(X[categorical_cols]),
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X.index  # Preserve index
)

# Combine encoded categoricals with numerical variables
X_processed = pd.concat([X_encoded, X[numerical_cols]], axis=1)

# Scale numerical variables
scaler = StandardScaler()
X_processed[numerical_cols] = scaler.fit_transform(X_processed[numerical_cols])

# Add LotFrontage to the dataset for imputation
X_processed["LotFrontage"] = df2_mnar["LotFrontage"]

# Initialize and apply KNN imputer
imputer = KNNImputer(n_neighbors=5, weights="uniform")
X_imputed = pd.DataFrame(imputer.fit_transform(X_processed), columns=X_processed.columns, index=X_processed.index)

# Inverse scaling for numerical variables
X_imputed[numerical_cols] = scaler.inverse_transform(X_imputed[numerical_cols])

# Decode categorical variables
decoded_categoricals = pd.DataFrame(
    encoder.inverse_transform(X_imputed[X_encoded.columns]),
    columns=categorical_cols,
    index=X_imputed.index
)

# Reconstruct DataFrame
df_imputed = pd.concat([decoded_categoricals, X_imputed[numerical_cols], X_imputed["LotFrontage"]], axis=1)

Evaluation:

In [ ]:
# Calculate R² score
r2 = r2_score(df2["LotFrontage"][df2_mnar["LotFrontage"].isna()], df_imputed["LotFrontage"][df2_mnar["LotFrontage"].isna()])
print("R-squared (R²) for the imputation:", r2)

R-squared (R²) for the imputation: -1.5949525899844113


###### **Conclusion**

In this section, we tested three different ways to handle missing data, each suited to a specific missing data type (MCAR, MAR, and MNAR).

For MasVnrType (MCAR), mode imputation led to an R² of -0.378, showing its limitations in preserving the original distribution. For FireplaceQu (MAR), predictive imputation with logistic regression performed slightly better, with an R² of -0.023, suggesting that using related variables helps improve accuracy. Finally, KNN imputation for LotFrontage (MNAR) had the lowest performance, with an R² of -1.595, indicating that imputing values based on nearest neighbors in high-dimensional spaces may not always be reliable.


##### **References**

- Use of chatGPT to generate regex expressions in Part 1 of assignment for the format type error section
